# Compare Experiments

Scans every completed run under `experiments/` (as produced by `dgd_training_demo.ipynb` or `run_experiments.ipynb`), ranks them by best val reconstruction loss, and compares loss curves / clustering quality / latent-space quality across them -- without re-running training or inference. Reads only what each run already persisted to disk (`config.yaml`, `models/best/training_results.pth`, `models/best/metadata.pth`, and, if present, `figures/latent/val/gmm_overview.png`).

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

experiments_dir = project_root / "experiments"


## Scan runs

A run folder (`experiments/<timestamp>_<experiment_name>/`) counts as *completed* if it has `config.yaml` at its root and `models/best/training_results.pth` + `models/best/metadata.pth` inside it -- written only once `DGDTrainer.train()` finishes (see `src/training/trainer.py`) -- and its metadata includes `best_recon_val_loss` (the model-selection criterion; a couple of the earliest runs in this repo predate that field). Anything missing those (still running, failed before finishing, or from before that schema) is skipped, with a note printed below.

In [ ]:
def load_run(run_dir: Path):
    """Load one run's config + persisted training history/metadata. Returns
    None if the run never finished, or predates best_recon_val_loss (see
    markdown above)."""
    config_path = run_dir / "config.yaml"
    best_dir = run_dir / "models" / "best"
    results_path = best_dir / "training_results.pth"
    metadata_path = best_dir / "metadata.pth"
    if not (config_path.is_file() and results_path.is_file() and metadata_path.is_file()):
        return None

    config = OmegaConf.load(config_path)
    history = torch.load(results_path, map_location="cpu")
    metadata = torch.load(metadata_path, map_location="cpu")

    if metadata.get("best_recon_val_loss", history.get("best_recon_val_loss")) is None:
        return None

    best_epoch = metadata.get("best_epoch", history.get("best_epoch"))
    # gmm_val_losses is 0.0 for every epoch before training.first_epoch_gmm
    # (see trainer.py) -- not missing, just genuinely zero until the GMM
    # prior turns on.
    gmm_val_losses = history.get("gmm_val_losses")
    best_gmm_val_loss = gmm_val_losses[best_epoch - 1] if gmm_val_losses else float("nan")
    gmm_train_losses = history.get("gmm_train_losses")
    best_gmm_train_loss = gmm_train_losses[best_epoch - 1] if gmm_train_losses else float("nan")
    recon_train_losses = history.get("recon_train_losses")
    best_recon_train_loss = recon_train_losses[best_epoch - 1] if recon_train_losses else float("nan")

    return {
        "run_dir": run_dir,
        "experiment_name": config.get("experiment_name", run_dir.name),
        "description": config.get("description", ""),
        "timestamp": run_dir.name.split("_ImageDGD")[0],
        "n_latent_copies": int(config.model.representation.get("n_latent_copies", 1)),
        "n_gmm_components": config.model.gmm.n_components,
        "latent_dim": config.model.representation.n_features,
        "epochs_run": len(history["train_losses"]),
        "epochs_configured": config.training.epochs,
        "best_epoch": best_epoch,
        "best_recon_train_loss": best_recon_train_loss,
        "best_recon_val_loss": metadata.get("best_recon_val_loss", history.get("best_recon_val_loss")),
        "best_gmm_train_loss": best_gmm_train_loss,
        "best_gmm_val_loss": best_gmm_val_loss,
        "best_val_loss": metadata.get("best_val_loss", history.get("best_val_loss")),
        "final_val_ami": history["val_ami_scores"][-1] if history.get("val_ami_scores") else float("nan"),
        "final_val_ari": history["val_ari_scores"][-1] if history.get("val_ari_scores") else float("nan"),
        "history": history,
    }


runs = []
skipped = []
for run_dir in sorted(experiments_dir.iterdir()):
    if not run_dir.is_dir():
        continue
    run = load_run(run_dir)
    if run is None:
        skipped.append(run_dir.name)
    else:
        runs.append(run)

print(f"Loaded {len(runs)} completed run(s).")
if skipped:
    print(f"Skipped {len(skipped)} incomplete/unfinished run(s):")
    for name in skipped:
        print(f"  - {name}")


## Summary table

Sorted by run timestamp. Columns that are identical across every loaded run (e.g. `epochs_run`, if every run happened to train the full schedule) are dropped automatically -- printed above the table when that happens -- since a constant column isn't a useful point of comparison. The loss-curve/AMI-ARI/latent-space cells below still *rank* by best val reconstruction loss for picking which runs to plot.

In [ ]:
summary_full = pd.DataFrame([{
    "experiment": r["experiment_name"],
    "timestamp": r["timestamp"],
    "best_recon_val": r["best_recon_val_loss"],
    "best_gmm_val": r["best_gmm_val_loss"],
    "best_val_loss": r["best_val_loss"],
    "final_val_AMI": r["final_val_ami"],
    "final_val_ARI": r["final_val_ari"],
    "best_epoch": r["best_epoch"],
    "epochs_run": r["epochs_run"],
    "n_latent_copies": r["n_latent_copies"],
    "gmm_k": r["n_gmm_components"],
    "latent_dim": r["latent_dim"],
    "description": r["description"],
} for r in runs]).sort_values("timestamp").reset_index(drop=True)

# Columns that are the same for every run aren't a useful point of
# comparison (e.g. epochs_run, if every run trained the full schedule) --
# dropped automatically rather than hardcoded, so this stays correct as
# EXPERIMENTS/config/experiment/*.yaml change over time. experiment/timestamp/
# description are kept regardless (identifiers/context, not "properties").
ALWAYS_KEEP = {"experiment", "timestamp", "description"}
constant_cols = [c for c in summary_full.columns
                 if c not in ALWAYS_KEEP and summary_full[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Dropping column(s) with the same value across every loaded run:")
    for c in constant_cols:
        print(f"  {c} = {summary_full[c].iloc[0]}")

summary = summary_full.drop(columns=constant_cols)

pd.set_option('display.max_colwidth', 60)
summary


## Loss curves

Val reconstruction loss per epoch, overlaid for the best `TOP_N` runs (by `best_recon_val_loss`) -- the number of runs worth showing on one plot is a judgment call, so it's a variable rather than hardcoded to "all of them".

In [ ]:
TOP_N = 8

ranked = sorted(runs, key=lambda r: r["best_recon_val_loss"])[:TOP_N]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for r in ranked:
    label = f"{r['experiment_name']} (best={r['best_recon_val_loss']:.4f})"
    axes[0].plot(r["history"]["recon_val_losses"], label=label, alpha=0.85)
    axes[1].plot(r["history"]["recon_train_losses"], label=label, alpha=0.85)

axes[0].set_title(f"Val reconstruction loss (top {len(ranked)})")
axes[1].set_title(f"Train reconstruction loss (top {len(ranked)})")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Reconstruction loss")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()


## Clustering quality (AMI / ARI)

Final (last GMM-refit-epoch) val AMI/ARI for every completed run, sorted by AMI.

In [ ]:
ami_ranked = sorted(runs, key=lambda r: r["final_val_ami"], reverse=True)
names = [r["experiment_name"] for r in ami_ranked]
ami_vals = [r["final_val_ami"] for r in ami_ranked]
ari_vals = [r["final_val_ari"] for r in ami_ranked]

x = range(len(names))
fig, ax = plt.subplots(figsize=(max(8, len(names) * 0.6), 5))
ax.bar([i - 0.2 for i in x], ami_vals, width=0.4, label="Val AMI")
ax.bar([i + 0.2 for i in x], ari_vals, width=0.4, label="Val ARI")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=60, ha='right', fontsize=8)
ax.set_ylabel("Score")
ax.set_title("Final val clustering quality by experiment")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## Best latent space, top runs

`figures/latent/val/gmm_overview.png` for the top 3 runs by val reconstruction loss, if that run's figures were already generated (see `generate_training_figures` in `src/visualization/report.py`) -- skipped per-run (with a note) if not.

In [ ]:
top3 = sorted(runs, key=lambda r: r["best_recon_val_loss"])[:3]

for r in top3:
    img_path = r["run_dir"] / "figures" / "latent" / "val" / "gmm_overview.png"
    if not img_path.is_file():
        print(f"{r['experiment_name']}: no figures/latent/val/gmm_overview.png on disk (figures not generated for this run) -- skipping")
        continue
    img = plt.imread(img_path)
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"{r['experiment_name']} -- best_recon_val={r['best_recon_val_loss']:.4f}, "
                 f"val AMI={r['final_val_ami']:.4f}, val ARI={r['final_val_ari']:.4f}")
    plt.tight_layout()
    plt.show()
